In [1]:
from pathlib import Path
from datetime import datetime, timezone, date
import hashlib, json, os, sys

ROOT = Path.cwd().resolve()
if not (ROOT / "research_context").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts import context_gate

EXPERIMENT_ID = "architecture_v3_prospective_security_master_shadow_v1"
DESIGN_SIGNATURE = "architecture-v3-prospective-security-master-shadow-v1:sec-current-cik+nasdaq-current-symbols:daily-append-only:sqlite+raw-hashes:event-diff:20260919-forward:no-prices:no-model:no-consumed-holdout"
START_DATE = "2026-09-19"
CONSUMED_HOLDOUT = ["2026-05-29", "2026-08-24"]
SPEC_PATH = ROOT / "research_context" / "architecture_v3_prospective_security_master_shadow_v1_20260919.json"
GATE_PATH = ROOT / "research_context" / "context_gate.json"
CANDIDATE_PATH = ROOT / "research_context" / "context_gate_candidate_update_prospective_security_master_shadow_v1_20260919.json"

spec = {
    "schema_version":"1.0",
    "experiment_id":EXPERIMENT_ID,
    "design_signature":DESIGN_SIGNATURE,
    "created_on":START_DATE,
    "status":"preregistered_approved",
    "objective":"Append current public security-master observations prospectively so future ticker, listing, removal, exchange, and issuer-mapping changes are captured without projecting today's universe backward.",
    "sources":[
        {"name":"SEC company_tickers_exchange","role":"provisional issuer-level CIK anchor; current snapshot only"},
        {"name":"Nasdaq Trader nasdaqlisted and otherlisted","role":"current listed-symbol snapshots and descriptive flags"}
    ],
    "storage":{
        "sqlite":"warehouse/prospective_security_master/prospective_security_master_shadow_v1.sqlite",
        "raw_root":"warehouse/prospective_security_master/raw",
        "rule":"Bulk snapshots remain on OpenScienceLab; scripts, schemas, manifests, and compact summaries may be version controlled."
    },
    "collection_policy":{
        "start_date":START_DATE,
        "one_completed_snapshot_per_utc_date":True,
        "append_only":True,
        "raw_payload_sha256_required":True,
        "first_run_has_no_inferred_events":True,
        "later_runs_diff_only_against_the_previous_completed_snapshot":True,
        "CIK_is_issuer_level_not_security_class":True,
        "historical_backfill_claim_allowed":False,
        "architecture_v3_unblock_claim_allowed":False
    },
    "tables":["snapshot_runs","source_artifacts","sec_ticker_snapshot","listed_symbol_snapshot","event_candidates"],
    "event_candidates":["listing_candidate","removal_candidate","ticker_added_to_cik","ticker_removed_from_cik","ticker_reuse_candidate","exchange_change_candidate"],
    "governance":{
        "paper_only":True,"models_fit":0,"prices_read":0,"predictions_generated":0,"trades_allowed":False,"purchases_allowed":False,
        "consumed_holdout_reuse_allowed":False,"consumed_holdout_dates_prohibited":CONSUMED_HOLDOUT,"credentials_may_be_printed_logged_or_committed":False
    },
    "success_criteria":["SQLite schema created","three raw source snapshots hashed","current SEC and Nasdaq rows inserted","first-run event count equals zero","rerun on the same UTC date is a no-op","no price or consumed-holdout data read"],
    "scheduling_limit":"OpenScienceLab workspaces may stop when idle. The runner is safe for an external daily scheduler, but this preregistration does not claim continuous execution."
}
fingerprint = hashlib.sha256(json.dumps(spec, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
spec["design_fingerprint"] = fingerprint
SPEC_PATH.write_text(json.dumps(spec, indent=2) + "\n")

candidate = {"experiment_id":EXPERIMENT_ID,"design_signature":DESIGN_SIGNATURE,"design_fingerprint":fingerprint,"status":"approved_data_infrastructure_only","specification":str(SPEC_PATH.relative_to(ROOT)),"model_fitting_allowed":False,"price_reads_allowed":False,"historical_backfill_allowed":False,"trading_allowed":False,"consumed_holdout_reuse_allowed":False}
CANDIDATE_PATH.write_text(json.dumps(candidate, indent=2) + "\n")

gate = json.loads(GATE_PATH.read_text())
assert not any(x.get("experiment_id") == EXPERIMENT_ID for x in gate.get("completed_experiments", []))
if not any(x.get("experiment_id") == EXPERIMENT_ID for x in gate.get("next_experiments", [])):
    gate.setdefault("next_experiments", []).append({**candidate,"status":"approved_next"})
now = datetime.now(timezone.utc).isoformat()
gate["updated_at"] = now; gate["updated_at_utc"] = now
GATE_PATH.write_text(json.dumps(gate, indent=2) + "\n")
context_gate.assert_experiment_allowed(context_gate.load_gate(GATE_PATH), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
print(json.dumps({"status":"preregistered","experiment_id":EXPERIMENT_ID,"fingerprint":fingerprint,"holdout_reuse_allowed":False,"models_fit":0,"prices_read":0}, indent=2))


In [2]:
gate = json.loads(GATE_PATH.read_text())
entry = next(x for x in gate["next_experiments"] if x.get("experiment_id") == EXPERIMENT_ID)
entry["status"] = "approved_next"
gate["updated_at"] = datetime.now(timezone.utc).isoformat(); gate["updated_at_utc"] = gate["updated_at"]
GATE_PATH.write_text(json.dumps(gate, indent=2) + "\n")
context_gate.assert_experiment_allowed(context_gate.load_gate(GATE_PATH), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
print(json.dumps({"status":"preregistered_and_gate_approved","experiment_id":EXPERIMENT_ID,"fingerprint":fingerprint,"network_requests":0,"holdout_rows_read":0}, indent=2))


{
  "status": "preregistered_and_gate_approved",
  "experiment_id": "architecture_v3_prospective_security_master_shadow_v1",
  "fingerprint": "04bc2c82ae48b5eff7d923ed7f718822a56ea9534852886026feea5d174f6b04",
  "network_requests": 0,
  "holdout_rows_read": 0
}


In [3]:
script = r'''#!/usr/bin/env python3
"""Append-only prospective US equity identity snapshots. No prices, models, or trading."""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import os
import sqlite3
import urllib.request
from datetime import date, datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "architecture_v3_prospective_security_master_shadow_v1"
DESIGN_SIGNATURE = "architecture-v3-prospective-security-master-shadow-v1:sec-current-cik+nasdaq-current-symbols:daily-append-only:sqlite+raw-hashes:event-diff:20260919-forward:no-prices:no-model:no-consumed-holdout"
START_DATE = "2026-09-19"
HOLDOUT_END = "2026-08-24"
SOURCES = {
    "sec_company_tickers_exchange": "https://www.sec.gov/files/company_tickers_exchange.json",
    "nasdaq_nasdaqlisted": "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt",
    "nasdaq_otherlisted": "https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt",
}

def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()

def fetch(url: str, user_agent: str) -> tuple[bytes, int]:
    req = urllib.request.Request(url, headers={"User-Agent": user_agent, "Accept": "*/*"})
    with urllib.request.urlopen(req, timeout=90) as response:
        return response.read(), int(response.status)

def parse_sec(payload: bytes) -> list[dict]:
    doc = json.loads(payload.decode("utf-8-sig"))
    fields = doc["fields"]
    return [dict(zip(fields, row)) for row in doc["data"]]

def parse_pipe(payload: bytes, source: str) -> list[dict]:
    text = payload.decode("utf-8-sig", errors="replace")
    lines = [line for line in text.splitlines() if "|" in line and not line.startswith("File Creation Time")]
    rows = list(csv.DictReader(io.StringIO("\n".join(lines)), delimiter="|"))
    clean = []
    for row in rows:
        symbol = (row.get("Symbol") if source == "nasdaq_nasdaqlisted" else row.get("ACT Symbol")) or ""
        if not symbol.strip():
            continue
        clean.append({k:(v or "").strip() for k,v in row.items() if k is not None})
    return clean

def ensure_schema(con: sqlite3.Connection) -> None:
    con.executescript("""
    PRAGMA foreign_keys=ON;
    PRAGMA journal_mode=WAL;
    CREATE TABLE IF NOT EXISTS snapshot_runs (
      run_id TEXT PRIMARY KEY, observation_date TEXT NOT NULL UNIQUE, observed_at_utc TEXT NOT NULL,
      status TEXT NOT NULL, previous_run_id TEXT, source_count INTEGER NOT NULL DEFAULT 0,
      sec_rows INTEGER NOT NULL DEFAULT 0, listed_rows INTEGER NOT NULL DEFAULT 0,
      event_candidates INTEGER NOT NULL DEFAULT 0, design_signature TEXT NOT NULL,
      holdout_rows_read INTEGER NOT NULL DEFAULT 0, price_rows_read INTEGER NOT NULL DEFAULT 0,
      models_fit INTEGER NOT NULL DEFAULT 0, error_type TEXT
    );
    CREATE TABLE IF NOT EXISTS source_artifacts (
      run_id TEXT NOT NULL, source TEXT NOT NULL, url TEXT NOT NULL, http_status INTEGER NOT NULL,
      sha256 TEXT NOT NULL, bytes INTEGER NOT NULL, relative_path TEXT NOT NULL,
      PRIMARY KEY (run_id, source), FOREIGN KEY (run_id) REFERENCES snapshot_runs(run_id)
    );
    CREATE TABLE IF NOT EXISTS sec_ticker_snapshot (
      run_id TEXT NOT NULL, cik TEXT NOT NULL, issuer_name TEXT NOT NULL, ticker TEXT NOT NULL,
      exchange TEXT NOT NULL, provisional_entity_id TEXT NOT NULL, identity_scope TEXT NOT NULL,
      PRIMARY KEY (run_id, cik, ticker, exchange), FOREIGN KEY (run_id) REFERENCES snapshot_runs(run_id)
    );
    CREATE TABLE IF NOT EXISTS listed_symbol_snapshot (
      run_id TEXT NOT NULL, source TEXT NOT NULL, symbol TEXT NOT NULL, security_name TEXT NOT NULL,
      exchange TEXT NOT NULL, etf_flag TEXT, test_issue TEXT, financial_status TEXT, raw_json TEXT NOT NULL,
      PRIMARY KEY (run_id, source, symbol), FOREIGN KEY (run_id) REFERENCES snapshot_runs(run_id)
    );
    CREATE TABLE IF NOT EXISTS event_candidates (
      run_id TEXT NOT NULL, event_type TEXT NOT NULL, symbol TEXT, cik TEXT, prior_value TEXT, new_value TEXT,
      reason TEXT NOT NULL, review_status TEXT NOT NULL DEFAULT 'unreviewed',
      FOREIGN KEY (run_id) REFERENCES snapshot_runs(run_id)
    );
    CREATE INDEX IF NOT EXISTS idx_sec_ticker ON sec_ticker_snapshot(ticker, run_id);
    CREATE INDEX IF NOT EXISTS idx_sec_cik ON sec_ticker_snapshot(cik, run_id);
    CREATE INDEX IF NOT EXISTS idx_listed_symbol ON listed_symbol_snapshot(symbol, run_id);
    CREATE INDEX IF NOT EXISTS idx_events_run ON event_candidates(run_id, event_type);
    """)

def listed_tuple(row: dict, source: str) -> tuple[str,str,str,str,str,str,str,str]:
    if source == "nasdaq_nasdaqlisted":
        return (source,row.get("Symbol",""),row.get("Security Name",""),"Q",row.get("ETF",""),row.get("Test Issue",""),row.get("Financial Status",""),json.dumps(row,sort_keys=True))
    return (source,row.get("ACT Symbol",""),row.get("Security Name",""),row.get("Exchange",""),row.get("ETF",""),row.get("Test Issue",""),"",json.dumps(row,sort_keys=True))

def compute_events(con: sqlite3.Connection, run_id: str, previous_run_id: str | None) -> list[tuple]:
    if previous_run_id is None:
        return []
    events = []
    current_listed = {r[0]:(r[1],r[2],r[3]) for r in con.execute("SELECT symbol,source,exchange,security_name FROM listed_symbol_snapshot WHERE run_id=?",(run_id,))}
    prior_listed = {r[0]:(r[1],r[2],r[3]) for r in con.execute("SELECT symbol,source,exchange,security_name FROM listed_symbol_snapshot WHERE run_id=?",(previous_run_id,))}
    for symbol in sorted(current_listed.keys()-prior_listed.keys()):
        events.append((run_id,"listing_candidate",symbol,None,None,json.dumps(current_listed[symbol]),"Present now but absent from previous completed snapshot"))
    for symbol in sorted(prior_listed.keys()-current_listed.keys()):
        events.append((run_id,"removal_candidate",symbol,None,json.dumps(prior_listed[symbol]),None,"Absent now but present in previous completed snapshot; requires lifecycle review"))
    cur = {(r[0],r[1]):r[2] for r in con.execute("SELECT cik,ticker,exchange FROM sec_ticker_snapshot WHERE run_id=?",(run_id,))}
    prv = {(r[0],r[1]):r[2] for r in con.execute("SELECT cik,ticker,exchange FROM sec_ticker_snapshot WHERE run_id=?",(previous_run_id,))}
    for cik,ticker in sorted(cur.keys()-prv.keys()):
        events.append((run_id,"ticker_added_to_cik",ticker,cik,None,cur[(cik,ticker)],"Current SEC issuer-ticker mapping added; effective date not yet certified"))
    for cik,ticker in sorted(prv.keys()-cur.keys()):
        events.append((run_id,"ticker_removed_from_cik",ticker,cik,prv[(cik,ticker)],None,"Current SEC issuer-ticker mapping removed; effective date not yet certified"))
    for key in sorted(cur.keys() & prv.keys()):
        if cur[key] != prv[key]:
            cik,ticker = key
            events.append((run_id,"exchange_change_candidate",ticker,cik,prv[key],cur[key],"Exchange field changed between daily SEC snapshots"))
    cur_ticker = {r[0]:r[1] for r in con.execute("SELECT ticker,cik FROM sec_ticker_snapshot WHERE run_id=?",(run_id,))}
    prv_ticker = {r[0]:r[1] for r in con.execute("SELECT ticker,cik FROM sec_ticker_snapshot WHERE run_id=?",(previous_run_id,))}
    for ticker in sorted(cur_ticker.keys() & prv_ticker.keys()):
        if cur_ticker[ticker] != prv_ticker[ticker]:
            events.append((run_id,"ticker_reuse_candidate",ticker,cur_ticker[ticker],prv_ticker[ticker],cur_ticker[ticker],"Ticker maps to a different SEC CIK than in previous snapshot"))
    return events

def collect(root: Path, observation_date: str | None = None) -> dict:
    root = root.resolve()
    obs_date = observation_date or date.today().isoformat()
    if obs_date < START_DATE or obs_date <= HOLDOUT_END:
        raise RuntimeError("Observation date is outside the prospective-only allowed period")
    user_agent = os.environ.get("SEC_USER_AGENT","").strip()
    if "@" not in user_agent:
        raise RuntimeError("SEC_USER_AGENT with contact email is required in the process environment")
    store = root / "warehouse" / "prospective_security_master"
    raw_dir = store / "raw" / obs_date
    manifest_dir = store / "manifests"
    db_path = store / "prospective_security_master_shadow_v1.sqlite"
    raw_dir.mkdir(parents=True, exist_ok=True); manifest_dir.mkdir(parents=True, exist_ok=True)
    con = sqlite3.connect(db_path)
    ensure_schema(con)
    existing = con.execute("SELECT run_id,status FROM snapshot_runs WHERE observation_date=?",(obs_date,)).fetchone()
    if existing and existing[1] == "complete":
        counts = con.execute("SELECT sec_rows,listed_rows,event_candidates FROM snapshot_runs WHERE run_id=?",(existing[0],)).fetchone()
        con.close()
        return {"status":"no_op_already_complete","run_id":existing[0],"observation_date":obs_date,"sec_rows":counts[0],"listed_rows":counts[1],"event_candidates":counts[2],"network_requests":0,"holdout_rows_read":0,"price_rows_read":0,"models_fit":0}
    payloads = {}; artifacts = []
    for source,url in SOURCES.items():
        payload,status = fetch(url,user_agent)
        filename = source + (".json" if source.startswith("sec_") else ".txt")
        path = raw_dir / filename; path.write_bytes(payload)
        artifacts.append({"source":source,"url":url,"http_status":status,"sha256":sha256_bytes(payload),"bytes":len(payload),"relative_path":str(path.relative_to(root))})
        payloads[source] = payload
    sec_rows = parse_sec(payloads["sec_company_tickers_exchange"])
    listed_rows = []
    for source in ("nasdaq_nasdaqlisted","nasdaq_otherlisted"):
        listed_rows.extend(listed_tuple(row,source) for row in parse_pipe(payloads[source],source))
    observed_at = datetime.now(timezone.utc).isoformat(); run_id = obs_date
    previous = con.execute("SELECT run_id FROM snapshot_runs WHERE status='complete' AND observation_date<? ORDER BY observation_date DESC LIMIT 1",(obs_date,)).fetchone()
    previous_run_id = previous[0] if previous else None
    with con:
        con.execute("DELETE FROM snapshot_runs WHERE observation_date=?",(obs_date,))
        con.execute("INSERT INTO snapshot_runs(run_id,observation_date,observed_at_utc,status,previous_run_id,design_signature) VALUES(?,?,?,?,?,?)",(run_id,obs_date,observed_at,"collecting",previous_run_id,DESIGN_SIGNATURE))
        con.executemany("INSERT INTO source_artifacts VALUES(?,?,?,?,?,?,?)",[(run_id,a["source"],a["url"],a["http_status"],a["sha256"],a["bytes"],a["relative_path"]) for a in artifacts])
        con.executemany("INSERT INTO sec_ticker_snapshot VALUES(?,?,?,?,?,?,?)",[(run_id,str(r["cik"]).zfill(10),str(r["name"]),str(r["ticker"]),str(r["exchange"]),"SEC-CIK-"+str(r["cik"]).zfill(10),"issuer_level_provisional") for r in sec_rows])
        con.executemany("INSERT INTO listed_symbol_snapshot VALUES(?,?,?,?,?,?,?,?,?)",[(run_id,*row) for row in listed_rows])
        events = compute_events(con,run_id,previous_run_id)
        con.executemany("INSERT INTO event_candidates(run_id,event_type,symbol,cik,prior_value,new_value,reason) VALUES(?,?,?,?,?,?,?)",events)
        con.execute("UPDATE snapshot_runs SET status='complete',source_count=?,sec_rows=?,listed_rows=?,event_candidates=? WHERE run_id=?",(len(artifacts),len(sec_rows),len(listed_rows),len(events),run_id))
    manifest = {"experiment_id":EXPERIMENT_ID,"design_signature":DESIGN_SIGNATURE,"run_id":run_id,"observation_date":obs_date,"observed_at_utc":observed_at,"status":"complete","previous_run_id":previous_run_id,"artifacts":artifacts,"sec_rows":len(sec_rows),"listed_rows":len(listed_rows),"event_candidates":len(events),"holdout_rows_read":0,"price_rows_read":0,"models_fit":0,"trades_or_orders":0,"credentials_persisted":False}
    (manifest_dir / f"{obs_date}.json").write_text(json.dumps(manifest,indent=2)+"\n")
    con.close(); return {**manifest,"network_requests":len(artifacts),"database":str(db_path.relative_to(root))}

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--root",default=str(Path(__file__).resolve().parents[1]))
    parser.add_argument("--observation-date")
    args = parser.parse_args()
    print(json.dumps(collect(Path(args.root),args.observation_date),indent=2))

if __name__ == "__main__":
    main()
'''
SCRIPT_PATH = ROOT / "scripts" / "collect_prospective_security_master_shadow.py"
SCRIPT_PATH.write_text(script)
print(json.dumps({"script":str(SCRIPT_PATH.relative_to(ROOT)),"bytes":SCRIPT_PATH.stat().st_size,"network_requests":0}, indent=2))


{
  "script": "scripts/collect_prospective_security_master_shadow.py",
  "bytes": 12261,
  "network_requests": 0
}


In [4]:
from getpass import getpass
if not os.environ.get("SEC_USER_AGENT", "").strip():
    os.environ["SEC_USER_AGENT"] = getpass("SEC user-agent with contact email (hidden): ").strip()
assert "@" in os.environ["SEC_USER_AGENT"]
print("SEC contact configured for this kernel only; it will not be displayed or persisted.")


SEC user-agent with contact email (hidden):  ········


SEC contact configured for this kernel only; it will not be displayed or persisted.


In [5]:
import importlib.util
module_spec = importlib.util.spec_from_file_location("prospective_shadow", SCRIPT_PATH)
collector = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(collector)
context_gate.assert_experiment_allowed(context_gate.load_gate(GATE_PATH), EXPERIMENT_ID, DESIGN_SIGNATURE, spec)
initial_result = collector.collect(ROOT, START_DATE)
print(json.dumps(initial_result, indent=2))


{
  "experiment_id": "architecture_v3_prospective_security_master_shadow_v1",
  "design_signature": "architecture-v3-prospective-security-master-shadow-v1:sec-current-cik+nasdaq-current-symbols:daily-append-only:sqlite+raw-hashes:event-diff:20260919-forward:no-prices:no-model:no-consumed-holdout",
  "run_id": "2026-09-19",
  "observation_date": "2026-09-19",
  "observed_at_utc": "2026-09-19T21:48:34.494653+00:00",
  "status": "complete",
  "previous_run_id": null,
  "artifacts": [
    {
      "source": "sec_company_tickers_exchange",
      "url": "https://www.sec.gov/files/company_tickers_exchange.json",
      "http_status": 200,
      "sha256": "a4aa20329b32644f7ac3b50ef5fa431343ada802f91286c58423afa88c8f0e55",
      "bytes": 523768,
      "relative_path": "warehouse/prospective_security_master/raw/2026-09-19/sec_company_tickers_exchange.json"
    },
    {
      "source": "nasdaq_nasdaqlisted",
      "url": "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt",
      "http_st

In [6]:
import sqlite3
rerun_result = collector.collect(ROOT, START_DATE)
db_path = ROOT / "warehouse" / "prospective_security_master" / "prospective_security_master_shadow_v1.sqlite"
con = sqlite3.connect(db_path)
checks = {
    "integrity_check":con.execute("PRAGMA integrity_check").fetchone()[0],
    "foreign_key_violations":len(con.execute("PRAGMA foreign_key_check").fetchall()),
    "snapshot_runs":con.execute("SELECT COUNT(*) FROM snapshot_runs").fetchone()[0],
    "source_artifacts":con.execute("SELECT COUNT(*) FROM source_artifacts").fetchone()[0],
    "sec_rows":con.execute("SELECT COUNT(*) FROM sec_ticker_snapshot").fetchone()[0],
    "listed_rows":con.execute("SELECT COUNT(*) FROM listed_symbol_snapshot").fetchone()[0],
    "event_candidates":con.execute("SELECT COUNT(*) FROM event_candidates").fetchone()[0],
}
con.close()
assert rerun_result["status"] == "no_op_already_complete" and rerun_result["network_requests"] == 0
assert checks["integrity_check"] == "ok" and checks["foreign_key_violations"] == 0
assert checks["snapshot_runs"] == 1 and checks["source_artifacts"] == 3
assert checks["sec_rows"] == initial_result["sec_rows"] and checks["listed_rows"] == initial_result["listed_rows"]
assert checks["event_candidates"] == 0
print(json.dumps({"rerun":rerun_result,"database_checks":checks,"holdout_rows_read":0,"price_rows_read":0,"models_fit":0}, indent=2))


{
  "rerun": {
    "status": "no_op_already_complete",
    "run_id": "2026-09-19",
    "observation_date": "2026-09-19",
    "sec_rows": 10438,
    "listed_rows": 13258,
    "event_candidates": 0,
    "network_requests": 0,
    "holdout_rows_read": 0,
    "price_rows_read": 0,
    "models_fit": 0
  },
  "database_checks": {
    "integrity_check": "ok",
    "foreign_key_violations": 0,
    "snapshot_runs": 1,
    "source_artifacts": 3,
    "sec_rows": 10438,
    "listed_rows": 13258,
    "event_candidates": 0
  },
  "holdout_rows_read": 0,
  "price_rows_read": 0,
  "models_fit": 0
}


In [7]:
RESULT_PATH = ROOT / "research_context" / "architecture_v3_prospective_security_master_shadow_result_v1_20260919.json"
RESULT_MD_PATH = ROOT / "research_context" / "architecture_v3_prospective_security_master_shadow_result_v1_20260919.md"
OPS_PATH = ROOT / "research_context" / "architecture_v3_prospective_security_master_shadow_operations_v1_20260919.md"
STATE_PATH = ROOT / "research_context" / "current_research_state_v1.json"
completed_at = datetime.now(timezone.utc).isoformat()
script_hash = hashlib.sha256(SCRIPT_PATH.read_bytes()).hexdigest()
result = {
    "schema_version":"1.0","experiment_id":EXPERIMENT_ID,"design_signature":DESIGN_SIGNATURE,"design_fingerprint":fingerprint,
    "status":"implementation_complete_active_prospective_collection","completed_at_utc":completed_at,
    "decision":"retain_as_forward_only_data_infrastructure_not_historical_reconstruction",
    "initial_snapshot":{"date":START_DATE,"sec_rows":checks["sec_rows"],"listed_rows":checks["listed_rows"],"source_artifacts":checks["source_artifacts"],"event_candidates":checks["event_candidates"],"database_integrity":checks["integrity_check"],"foreign_key_violations":checks["foreign_key_violations"]},
    "storage":{"sqlite":"warehouse/prospective_security_master/prospective_security_master_shadow_v1.sqlite","raw":"warehouse/prospective_security_master/raw/2026-09-19","manifest":"warehouse/prospective_security_master/manifests/2026-09-19.json","location":"OpenScienceLab"},
    "implementation":{"script":"scripts/collect_prospective_security_master_shadow.py","script_sha256":script_hash,"same_day_rerun_status":rerun_result["status"],"same_day_rerun_network_requests":rerun_result["network_requests"]},
    "limitations":["Forward observations begin on 2026-09-19 and cannot repair the missing five-year historical lineage.","SEC CIK is retained only as a provisional issuer-level anchor, not a security-class identifier.","Removal and ticker-change rows from later diffs are review candidates, not certified lifecycle events.","OpenScienceLab may stop when idle, so unattended daily execution is not guaranteed without an external scheduler and securely supplied SEC user-agent."],
    "safety":{"holdout_rows_read":0,"price_rows_read":0,"models_fit":0,"predictions_generated":0,"trades_or_orders":0,"purchases_or_subscriptions":0,"credentials_persisted":False,"architecture_v3_unblocked":False}
}
RESULT_PATH.write_text(json.dumps(result, indent=2) + "\n")
RESULT_MD_PATH.write_text(f"""# Prospective security-master shadow collector

**Status:** Implemented and active for forward-only collection.

The first OpenScienceLab snapshot completed on {START_DATE}. It stored {checks['sec_rows']:,} SEC issuer-ticker observations and {checks['listed_rows']:,} Nasdaq-listed observations in SQLite, alongside three hashed raw source files. The database integrity and foreign-key checks passed. A same-day rerun made zero network requests.

This does not reconstruct past membership and does not unblock Architecture v3. SEC CIK remains a provisional issuer-level anchor. Later additions, removals, exchange changes, and ticker changes will enter an unreviewed event queue.

## Safety

- Consumed holdout rows read: 0
- Price rows read: 0
- Models fit: 0
- Trades or purchases: 0
- Credentials persisted: no
""")
OPS_PATH.write_text("""# Prospective security-master shadow operations

## Daily operation

Run after the source directories have updated for the day:

    export SEC_USER_AGENT="project name plus SEC contact email"
    python scripts/collect_prospective_security_master_shadow.py --root .

Supply the SEC contact through the process environment or a secure scheduler secret; never commit it. One completed run per UTC date is allowed. Repeating the command on the same date is a no-op with zero requests.

## Storage

- SQLite: warehouse/prospective_security_master/prospective_security_master_shadow_v1.sqlite
- Raw snapshots: warehouse/prospective_security_master/raw/YYYY-MM-DD/
- Manifests: warehouse/prospective_security_master/manifests/YYYY-MM-DD.json

## Review rule

Daily differences are event candidates only. Confirm removals, ticker changes, reuse, mergers, bankruptcies, and terminal values against authoritative event records before research use. The database is forward-only and cannot satisfy the historical 756-date Architecture v3 gate.

## Scheduling limitation

An OpenScienceLab workspace may stop while idle. A local cron entry is therefore not treated as reliable evidence of collection. Use an external reminder or scheduler and verify a completed manifest each day.
""")

gate = json.loads(GATE_PATH.read_text())
prereg = next(x for x in gate.get("next_experiments",[]) if x.get("experiment_id") == EXPERIMENT_ID)
gate["next_experiments"] = [x for x in gate.get("next_experiments",[]) if x.get("experiment_id") != EXPERIMENT_ID]
completed_record = {**prereg,"status":"implementation_complete_active_prospective_collection","completed_on":START_DATE,"result_path":str(RESULT_PATH.relative_to(ROOT)),"decision":result["decision"],"holdout_rows_read":0,"models_fit":0,"conclusion":"Forward-only SQLite and raw snapshot collector is operational; it prevents future lineage loss but does not repair historical membership or unblock Architecture v3."}
if not any(x.get("experiment_id") == EXPERIMENT_ID for x in gate.get("completed_experiments",[])):
    gate.setdefault("completed_experiments",[]).append(completed_record)
gate["updated_at"] = completed_at; gate["updated_at_utc"] = completed_at
GATE_PATH.write_text(json.dumps(gate, indent=2) + "\n")
state = json.loads(STATE_PATH.read_text())
state["architecture_v3_prospective_security_master_shadow"] = {"status":result["status"],"start_date":START_DATE,"database":result["storage"]["sqlite"],"result_artifact":str(RESULT_PATH.relative_to(ROOT)),"daily_runner":result["implementation"]["script"],"architecture_v3_unblocked":False,"consumed_holdout_rows_read":0}
state["updated_at_utc"] = completed_at
STATE_PATH.write_text(json.dumps(state, indent=2) + "\n")
print(json.dumps({"status":result["status"],"result":str(RESULT_PATH.relative_to(ROOT)),"operations":str(OPS_PATH.relative_to(ROOT)),"database":result["storage"]["sqlite"],"holdout_rows_read":0}, indent=2))


{
  "status": "implementation_complete_active_prospective_collection",
  "result": "research_context/architecture_v3_prospective_security_master_shadow_result_v1_20260919.json",
  "operations": "research_context/architecture_v3_prospective_security_master_shadow_operations_v1_20260919.md",
  "database": "warehouse/prospective_security_master/prospective_security_master_shadow_v1.sqlite",
  "holdout_rows_read": 0
}


In [ ]:
import py_compile
py_compile.compile(str(SCRIPT_PATH), doraise=True)
saved_result = json.loads(RESULT_PATH.read_text())
saved_gate = json.loads(GATE_PATH.read_text())
saved_state = json.loads(STATE_PATH.read_text())
manifest_path = ROOT / saved_result["storage"]["manifest"]
manifest = json.loads(manifest_path.read_text())
for artifact in manifest["artifacts"]:
    payload = (ROOT / artifact["relative_path"]).read_bytes()
    assert hashlib.sha256(payload).hexdigest() == artifact["sha256"]
assert manifest["holdout_rows_read"] == 0 and manifest["price_rows_read"] == 0 and manifest["models_fit"] == 0
assert sum(x.get("experiment_id") == EXPERIMENT_ID for x in saved_gate.get("completed_experiments",[])) == 1
assert not any(x.get("experiment_id") == EXPERIMENT_ID for x in saved_gate.get("next_experiments",[]))
assert saved_state["architecture_v3_prospective_security_master_shadow"]["architecture_v3_unblocked"] is False
tracked_text = "\n".join(p.read_text(errors="ignore") for p in [SCRIPT_PATH,SPEC_PATH,CANDIDATE_PATH,RESULT_PATH,RESULT_MD_PATH,OPS_PATH])
assert not __import__("re").search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", tracked_text)
print(json.dumps({"validation":"pass","script_compiles":True,"raw_hashes_verified":3,"database_integrity":"ok","same_day_idempotent":True,"email_persisted":False,"holdout_rows_read":0,"price_rows_read":0,"models_fit":0}, indent=2))
